# Evaluation demo

Load validated result artifacts, inspect repository-balanced summaries, and evaluate a hard-voting ensemble.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
from nlbse24.evaluation.analysis import analyze_results
from nlbse24.evaluation.results import load_artifacts

In [ ]:
tables = analyze_results(ROOT / 'results')
tables['models']

In [ ]:
tables['repositories']

## Optional hard-voting ensemble

Only align artifacts with the same protocol, repository, fold, seed, and test fingerprint.

In [ ]:
from collections import defaultdict
from nlbse24.evaluation.ensemble import evaluate_ensemble
models = ['tfidf_logistic_regression', 'setfit_mpnet']
artifacts = [a for a in load_artifacts(ROOT / 'results') if a['protocol'] == 'cv' and a['model']['name'] in models]
by_key = defaultdict(dict)
for artifact in artifacts:
    key = (artifact['repository'], artifact['seed'], artifact['fold'], artifact['data']['test_fingerprint'])
    by_key[key][artifact['model']['name']] = artifact
groups = [[items[model] for model in models] for items in by_key.values() if set(items) == set(models)]
evaluate_ensemble(groups, strategy='hard')['cross_repository_macro_f1']